In [17]:
#load the test graph data

import joblib


graphs = joblib.load("/data2/r10user3/Spatial_Gene_Cell_Ratio/code/graphs/human_lung_cell2location_high250gene_celltype80_KimiaNet_graphs")
test_slides = open("/data2/r10user3/Spatial_Gene_Cell_Ratio/code/test_leave1out.txt").read().split('\n')
test_loader = dict()
for item in test_slides:
    test_loader[item] = graphs[item]

In [18]:
test_loader[item].keys()

dict_keys(['features', 'labels', 'adj', 'names'])

In [26]:
# create model and load weights

from GCN_models import GCN
import os
import torch

gpu_list = [5]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = GCN(input_dim=1024, output_dim=250+80)
# model = GCN(input_dim=1024, output_dim=250)
model = GCN(input_dim=1024, output_dim=80)

# checkpoint = torch.load("/data2/r10user3/Spatial_Gene_Cell_Ratio/code/model_weights/cell_gene_2layersimpleGNN_lr1e-4_best_cell_all_abundance_average.pth")
# checkpoint = torch.load("/data2/r10user3/Spatial_Gene_Cell_Ratio/code/model_weights/cell_gene_2layersimpleGNN_lr1e-4_best_gene_all_average.pth")
checkpoint = torch.load("/data2/r10user3/Spatial_Gene_Cell_Ratio/code/model_weights/only_cell_2layersimpleGNN_lr1e-4_best_cell_all_abundance_average.pth")


model.load_state_dict(checkpoint)
model = model.to(device)

In [31]:
# inference 

Predictions = dict()

with torch.no_grad():
    model.eval()
    
    for item in test_loader:
        feature = torch.from_numpy(graphs[item]['features']).to(device).unsqueeze(0)
        label = torch.from_numpy(graphs[item]['labels']).to(device).unsqueeze(0)
        adj = torch.from_numpy(graphs[item]['adj']).to(torch.float32).to(device).unsqueeze(0)
        mask = torch.ones((feature.shape[1])).to(device).unsqueeze(0)

        gene_label = label[:, :, :250]
        cell_abundance_label = label[:, :, 250:]
        
        output = model(node_feat=feature, adj=adj, mask=mask)

        # gene_output = output[:, :, :250]
        # cell_abundance_output = output[:, :, 250:]
        
        cell_abundance_output = output[:, :, :]
        
        test_label_array = label.squeeze().cpu().detach().numpy()
        test_prediction_array = output.squeeze().cpu().detach().numpy()
        test_names = graphs[item]['names']
        
        Predictions[item] = {
            # 'gene_predictions': gene_output.squeeze().cpu().detach().numpy(),
            # 'gene_labels': gene_label.squeeze().cpu().detach().numpy(),
            'cell_abundance_predictions': cell_abundance_output.squeeze().cpu().detach().numpy(),
            'cell_abundance_labels': cell_abundance_label.squeeze().cpu().detach().numpy(),
            'spot_names': test_names
        }


In [32]:
Predictions.keys()

dict_keys(['WSA_LngSP8759311', 'WSA_LngSP8759312', 'WSA_LngSP8759313'])

In [35]:
Predictions['WSA_LngSP8759311'].keys()

dict_keys(['cell_abundance_predictions', 'cell_abundance_labels', 'spot_names'])

In [36]:
# save the inferenced results

# save_path = '/data2/r10user3/Spatial_Gene_Cell_Ratio/code/predictions/cell_gene_2layersimpleGNN_lr1e-4_best_cell_all_abundance_average.pkl'
# save_path = '/data2/r10user3/Spatial_Gene_Cell_Ratio/code/predictions/cell_gene_2layersimpleGNN_lr1e-4_best_gene_all_average.pkl'
# save_path = '/data2/r10user3/Spatial_Gene_Cell_Ratio/code/predictions/only_gene_2layersimpleGNN_lr1e-3_best_gene_all_average.pkl'
save_path = '/data2/r10user3/Spatial_Gene_Cell_Ratio/code/predictions/only_cell_2layersimpleGNN_lr1e-4_best_cell_all_abundance_average.pkl'

joblib.dump(Predictions, save_path)

['/data2/r10user3/Spatial_Gene_Cell_Ratio/code/predictions/only_cell_2layersimpleGNN_lr1e-4_best_cell_all_abundance_average.pkl']